# Multimodal Traffic Count Data - Silver Layer

## Objective
Transform, cleanse, and standardize raw Paris multimodal traffic count data from bronze.bronze_multimodal to build a clean, production-ready Silver Delta table (silver.silver_multimodal).

## Data Flow
bronze.bronze_multimodal → Spark DataFrame → silver.silver_multimodal

## Source
The underlying data comes from the Paris OpenData API: Comptage multimodal - Données de comptage en temps réel.

## Input
Bronze Delta table: bronze.bronze_multimodal

## Output
Silver Delta table: silver.silver_multimodal

## Silver Layer Principle
The Silver layer cleanses, enforces schema integrity, and standardizes data structures for downstream analytics. Column names are mapped from French/technical identifiers to standardized English snake_case names, categorical string values (transport mode, lane type) are translated into English, date/time fields are cast to appropriate temporal types, and null value distributions are audited prior to persistence.

## Processing Steps
1. **Load Bronze Data:** Read raw table into PySpark and verify row count.
2. **Standardize Column Names:** Rename columns to English snake_case.
3. **Value Translation:** Translate transport_mode and lane_type values to English.
4. **Data Type Conversions:** Cast timestamp string to TimestampType.
5. **Data Quality & Null Auditing:** Count null values across all columns.
6. **Write to Silver:** Save cleaned DataFrame to silver.silver_multimodal Delta table.

In [0]:
# Importing libraries
from pyspark.sql.functions import col, count, when
from pyspark.sql import functions as F

# LOAD BRONZE DATA

In [0]:
# Load data from bronze schema
df_bronze_multimodal=spark.table("workspace.bronze.bronze_multimodal")

In [0]:
# Inspecting the schema 
df_bronze_multimodal.printSchema()

In [0]:
# count number of rows in `df_bronze_multimodal` 
df_count=df_bronze_multimodal.count()
print(df_count)

In [0]:
# display the dataframe 
df_bronze_multimodal.limit(10).display()

# CLEAN DATA  AND CHECK FOR NULLS

In [0]:
# Mapping raw Multimodal column names to standardized English snake_case names
multimodal_column_mapping = {
    "id_trajectoire": "trajectory_id",
    "id_site": "site_id",
    "label": "site_label",
    "t": "timestamp",
    "mode": "transport_mode",
    "nb_usagers": "user_count",
    "voie": "lane_type",
    "sens": "direction",
    "trajectoire": "trajectory_direction",
    "coordonnees_geo": "geo_coordinates",
    "_ingestion_timestamp": "_ingestion_timestamp",
}

# Initialize df_multimodal before looping
df_multimodal = df_bronze_multimodal

# Dynamically rename columns in PySpark
for old_col, new_col in multimodal_column_mapping.items():
    df_multimodal = df_multimodal.withColumnRenamed(old_col, new_col)

In [0]:
# Display the new column names 
df_multimodal.limit(10).display()

In [0]:
# Retrieve distinct transport modes
df_multimodal.select('transport_mode').distinct().display()

In [0]:
# Retrieve distinct lane types
df_multimodal.select('lane_type').distinct().display()

In [0]:
# Translate transport_mode values to English
df_multimodal = df_multimodal.withColumn(
    "transport_mode",
    F.when(F.col("transport_mode") == "Trottinettes", "Scooters")
     .when(F.col("transport_mode") == "Vélos", "Bicycles")
     .when(F.col("transport_mode") == "2 roues motorisées", "Motorized Two-Wheelers")
     .when(F.col("transport_mode") == "Véhicules légers < 3,5t", "Light Vehicles (< 3.5t)")
     .otherwise(F.col("transport_mode"))
)

# Translate lane_type values to English
df_multimodal = df_multimodal.withColumn(
    "lane_type",
    F.when(F.col("lane_type") == "Piste cyclable", "Bicycle Lane")
     .when(F.col("lane_type") == "Coronapiste", "Temporary Bike Lane")
     .when(F.col("lane_type") == "Voie de circulation générale", "General Traffic Lane")
     .otherwise(F.col("lane_type"))
)

In [0]:
# Display the new data frame 
df_multimodal.limit(10).display()

In [0]:
# Cast timestamp column to timestamp 
df_multimodal = df_multimodal.withColumn(
    "timestamp",
    F.to_timestamp(F.col("timestamp"), "yyyy-MM-dd'T'HH:mm:ssXXX")
)
df_multimodal.limit(10).display()

In [0]:
# Check for nulls
null_counts_df = df_multimodal.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in df_multimodal.columns
])

# Display the summary table showing NULL count per column
display(null_counts_df)

# WRITE TO SILVER LAYER

In [0]:
df_multimodal\
    .write\
        .format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true") \
        .saveAsTable("silver.silver_multimodal")

# CHECKING THE SILVER TABLE

In [0]:
%sql 
SELECT *
FROM workspace.silver.silver_multimodal
LIMIT 10 ;